In [7]:
# Cell: Process raw network logs into ML-ready CSVs with correct Buffer Level parsing
import os
import re
import pandas as pd

base_dir   = "QoE_train"                          # root containing set-folders
output_dir = os.path.join(base_dir, "Net_train")
os.makedirs(output_dir, exist_ok=True)

# pick up any directory whose name contains 'set' followed by a digit
set_pattern = re.compile(r'.*set[1-4].*', re.IGNORECASE)
sets = [
    d for d in os.listdir(base_dir)
    if os.path.isdir(os.path.join(base_dir, d)) and set_pattern.match(d)
]

# Regex patterns
tp_pattern    = re.compile(r"Segment\s+(\d+)\s+Throughput:\s*([\d.E+]+)", re.IGNORECASE)
br_pattern    = re.compile(r"Segment\s+(\d+).*?(\d+(?:\.\d+)?)\s*kbps",         re.IGNORECASE)
rebuf_pattern = re.compile(r"Segment:\s*(\d+).*?Duration:\s*([\d.]+)s",       re.IGNORECASE)

for s in sets:
    folder = os.path.join(base_dir, s)
    files  = os.listdir(folder)
    tp_file    = next(f for f in files if f.lower().startswith("throughput"))
    buf_file   = next(f for f in files if f.lower().startswith("bufferlogger"))
    br_file    = next(f for f in files if f.lower().startswith("corrected_bitrate"))
    rebuf_file = next(f for f in files if f.lower().startswith("rebuffering"))

    # parse into maps
    tp_map, buf_map, br_map, rebuf_map = {}, {}, {}, {}

    # Buffer sizes
    for line in open(os.path.join(folder, buf_file)):
        if line.strip().startswith("Segment"):
            parts = line.split("Buffer Size:")
            if len(parts) == 2:
                seg = int(parts[0].split()[1])
                buf_map[seg] = float(parts[1].strip())

    # Throughput → kbps
    # if throughput is higher than 85 Mbps, cap it at 85 Mbps
    for line in open(os.path.join(folder, tp_file)):
        m = tp_pattern.search(line)
        if m:
            tp_map[int(m.group(1))] = min(float(m.group(2)) / 1e3, 85000.0)

    # Bitrate (kbps)
    for line in open(os.path.join(folder, br_file)):
        m = br_pattern.search(line)
        if m:
            br_map[int(m.group(1))] = float(m.group(2))

    # Rebuffer durations
    for line in open(os.path.join(folder, rebuf_file)):
        m = rebuf_pattern.search(line)
        if m:
            rebuf_map[int(m.group(1))] = float(m.group(2))

    # determine max segment index
    max_seg = max(
        *map(lambda m: max(m.keys(), default=0), 
             [tp_map, buf_map, br_map, rebuf_map])
    )

    # build rows
    rows = []
    for seg in range(1, max_seg + 1):
        # fixed-length history of previous 8 throughputs (kbps)
        history = [tp_map.get(i, 0.0) for i in range(seg - 8, seg)]
        rows.append({
            "Segment Number":     seg,
            "Buffer Level":       buf_map.get(seg,    0.0),
            "Bitrate (kbps)":     br_map.get(seg,    0.0),
            "Throughput History": history,
            "Rebuffer Time":      rebuf_map.get(seg, 0.0),
        })

    # write CSV
    df_out = pd.DataFrame(rows)
    # use only the digit at end of the folder name to label file
    set_id = re.search(r"set([1-4])", s, re.IGNORECASE).group(1)
    out_path = os.path.join(output_dir, f"processed_segments_{set_id}.csv")
    df_out.to_csv(out_path, index=False)
    print(f"[INFO] Wrote {out_path} ({len(rows)} segments)")


[INFO] Wrote QoE_train/Net_train/processed_segments_4.csv (2114 segments)
[INFO] Wrote QoE_train/Net_train/processed_segments_3.csv (2114 segments)
[INFO] Wrote QoE_train/Net_train/processed_segments_2.csv (2114 segments)
[INFO] Wrote QoE_train/Net_train/processed_segments_1.csv (2114 segments)


In [8]:
# Cell 2: Collect Eye files into Eye_train folder
import os
import re
import shutil

base_dir        = "QoE_train"
eye_train_dir   = os.path.join(base_dir, "Eye_train")
os.makedirs(eye_train_dir, exist_ok=True)

# dynamically discover any directory whose name contains 'set' followed by a digit
set_pattern = re.compile(r'.*(set)(\d).*', re.IGNORECASE)
sets = [
    d for d in os.listdir(base_dir)
    if os.path.isdir(os.path.join(base_dir, d)) and set_pattern.match(d)
]

for s in sets:
    m = set_pattern.match(s)
    set_id = m.group(2)  # the digit after 'set'
    folder = os.path.join(base_dir, s)

    # find the eye-tracking CSV
    eye_file = next(
        f for f in os.listdir(folder)
        if f.lower().startswith("eyetrackingdata") and f.lower().endswith(".csv")
    )

    src = os.path.join(folder, eye_file)
    dst_name = f"EyeTrackingData_set{set_id}.csv"
    dst = os.path.join(eye_train_dir, dst_name)

    shutil.copy(src, dst)
    print(f"[INFO] Copied {eye_file} → Eye_train/{dst_name}")


[INFO] Copied EyeTrackingData_video0.csv → Eye_train/EyeTrackingData_set4.csv
[INFO] Copied EyeTrackingData_video0.csv → Eye_train/EyeTrackingData_set3.csv
[INFO] Copied EyeTrackingData_video0.csv → Eye_train/EyeTrackingData_set2.csv
[INFO] Copied EyeTrackingData_video0.csv → Eye_train/EyeTrackingData_set1.csv


In [9]:
# import os
# import pandas as pd

# # -- 1) Locate the single CSV in ./Data3 --
# data_dir = "./Data3"
# csv_files = [f for f in os.listdir(data_dir) if f.lower().endswith(".csv")]
# if len(csv_files) != 1:
#     raise RuntimeError(f"Expected exactly one CSV in {data_dir}, found: {csv_files}")
# file_path = os.path.join(data_dir, csv_files[0])

# # -- 2) Load --
# df = pd.read_csv(file_path)
# print(f"[Load] {file_path}: {df.shape[0]} rows, {df.shape[1]} columns")

# # -- 3) Drop columns where ≥96% of values are identical --
# threshold = 0.96
# to_drop = []
# for col in df.columns:
#     top_pct = df[col].value_counts(normalize=True, dropna=False).iloc[0]
#     if top_pct >= threshold:
#         to_drop.append(col)
# if to_drop:
#     df = df.drop(columns=to_drop)
# print(f"[Filter] Dropped {len(to_drop)} near-constant columns: {to_drop}")
# print(f"[Filter] Remaining columns: {df.shape[1]}")

# # -- 4) Fill any NaN with 0 --
# df = df.fillna(0)
# print(f"[Clean] NaNs replaced with 0")

# # -- 5) Overwrite the original CSV in-place --
# df.to_csv(file_path, index=False)
# print(f"[Save] Original file overwritten at {file_path}")
